In [1]:
!pip install "jax[cuda12]"

  Using cached jax-0.7.2-py3-none-any.whl.metadata (13 kB)
  Using cached jaxlib-0.7.2-cp312-cp312-manylinux_2_27_x86_64.whl.metadata (1.3 kB)
  Using cached ml_dtypes-0.5.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.9 kB)
  Using cached numpy-2.3.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached scipy-1.16.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (62 kB)
  Using cached jax_cuda12_plugin-0.7.2-cp312-cp312-manylinux_2_27_x86_64.whl.metadata (2.0 kB)
  Using cached jax_cuda12_pjrt-0.7.2-py3-none-manylinux_2_27_x86_64.whl.metadata (579 bytes)
  Using cached nvidia_cublas_cu12-12.9.1.4-py3-none-manylinux_2_27_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_cupti_cu12-12.9.79-py3-none-manylinux_2_25_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_cuda_nvcc_cu12-12.9.86-py3-none-manylinux2010_x86_64.manylinux_2_12_

In [11]:
import jax
import jaxlib

print(f"JAX version: {jax.__version__}")
print(f"jaxlib version: {jaxlib.__version__}")

JAX version: 0.7.2
jaxlib version: 0.7.2


In [12]:
%env MUJOCO_GL=egl

env: MUJOCO_GL=egl


In [16]:
!pip install tensorflow
!pip install tf2onnx==1.16.1
!pip install orbax-checkpoint==0.11.25
!pip install onnx==1.17.0
!pip install onnxruntime==1.22.1

  Using cached tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.5 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached gast-0.6.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-manylinux2010_x86_64.whl.metadata (5.2 kB)
  Using cached protobuf-6.32.1-cp39-abi3-manylinux2014_x86_64.whl.metadata (593 bytes)
  Using cached termcolor-3.1.0-py3-none-any.whl.metadata (6.4 kB)
  Using cached wrapt-1.17.3-cp312-cp312-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl.metadata (6.4 kB)
  Using cached grpcio-1.75.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (3.7 kB)
  Using cached tensorboard-2.20.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached keras-3.11.3-py3-none-any.whl.metadata (5.9 kB)
  Using cached h5py-3.14.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl

In [18]:
# -*- coding: utf-8 -*-

# standard libraries
import os
# ---- 互換・ログ抑制（必ず import 前に）----
os.environ["JAX_PLATFORMS"] = "cpu"                   # GPUなし環境の安全運転
os.environ["JAX_EXPORT_CALLING_CONVENTION_VERSION"] = "9"  # TFがv10未対応でもOK
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"              # TF 警告抑制

# 3rd party libraries
from pathlib import Path
import numpy as np

import tf2onnx

import numpy as np
import tensorflow as tf
import tf2onnx
import onnxruntime as ort
import jax
import jax.numpy as jnp
import orbax.checkpoint as ocp
from pathlib import Path
from jax.experimental import jax2tf

# === NumPy 2.0 互換パッチ: np.cast をエミュレート（tf2onnx の定数畳み込みが使用）===
if not hasattr(np, "cast"):
    class _CastDict(dict):
        pass
    np.cast = _CastDict()

tf.get_logger().setLevel("ERROR")
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)

class OnnxConvert:

    def __init__(
            self,
            checkpoint_dir_path,
            onnx_file_path,
            obs_dim=68,
            act_dim=10,
            policy_hidden_layer_sizes=(128, 128, 128, 128),
            value_hidden_layer_sizes=(256, 256, 256, 256, 256)):
        """Initialize ONNX converter for JAX/Orbax checkpoints.

        Args:
            checkpoint_dir_path: Path to Orbax checkpoint directory.
            onnx_file_path: Output ONNX file path.
            obs_dim: Observation dimension.
            act_dim: Action dimension.
            policy_hidden_layer_sizes: Tuple of policy MLP hidden layer sizes.
            value_hidden_layer_sizes: Tuple of value MLP hidden layer sizes.
        """
        self.checkpoint_dir_path = checkpoint_dir_path
        self.onnx_file_path = onnx_file_path
        self.obs_dim = obs_dim
        self.act_dim = act_dim
        self.policy_hidden_layer_sizes = policy_hidden_layer_sizes
        self.value_hidden_layer_sizes = value_hidden_layer_sizes
        # Ensure output directory exists
        Path(Path(onnx_file_path).parent).mkdir(parents=True, exist_ok=True)

        self.normalizer_spec = {
            "mean":             jax.ShapeDtypeStruct((self.obs_dim,), jnp.float32),
            "std":              jax.ShapeDtypeStruct((self.obs_dim,), jnp.float32),
            "count":            jax.ShapeDtypeStruct((),          jnp.float32),
            "summed_variance":  jax.ShapeDtypeStruct((self.obs_dim,), jnp.float32),
        }

        self.policy_spec = {"params": {}}
        in_dim = self.obs_dim
        for i, h in enumerate(self.policy_hidden_layer_sizes):
            self.policy_spec["params"][f"hidden_{i}"] = {
                "kernel": jax.ShapeDtypeStruct((in_dim, h), jnp.float32),
                "bias":   jax.ShapeDtypeStruct((h,),        jnp.float32),
            }
            in_dim = h
        self.policy_spec["params"][f"hidden_{len(self.policy_hidden_layer_sizes)}"] = {
            "kernel": jax.ShapeDtypeStruct((in_dim, self.act_dim), jnp.float32),
            "bias":   jax.ShapeDtypeStruct((self.act_dim,),        jnp.float32),
        }

        self.value_spec = {"params": {}}
        in_dim = self.obs_dim
        for i, h in enumerate(self.value_hidden_layer_sizes):
            self.value_spec["params"][f"hidden_{i}"] = {
                "kernel": jax.ShapeDtypeStruct((in_dim, h), jnp.float32),
                "bias":   jax.ShapeDtypeStruct((h,),        jnp.float32),
            }
            in_dim = h
        self.value_spec["params"][f"hidden_{len(self.value_hidden_layer_sizes)}"] = {
            "kernel": jax.ShapeDtypeStruct((in_dim, 1), jnp.float32),
            "bias":   jax.ShapeDtypeStruct((1,),        jnp.float32),
        }

        self.item_spec = (self.normalizer_spec, self.policy_spec, self.value_spec)

        self.cpu = jax.devices("cpu")[0]
        self.single_cpu = jax.sharding.SingleDeviceSharding(self.cpu)
        self.restore_args = jax.tree_util.tree_map(lambda _: ocp.ArrayRestoreArgs(sharding=self.single_cpu),
                                                   self.item_spec)

        self.normalizer_stats = None
        self.policy_params = None
        self.value_params = None
        self.p_keys = None
        self.det_obs = None
        self.det_act = None

    def _restore_checkpoint(self):
        """Restore parameters from Orbax checkpoint to class fields."""
        checkpointer = ocp.Checkpointer(ocp.PyTreeCheckpointHandler())
        # Restore normalizer, policy, and value parameters
        normalizer_stats, policy_tree, value_tree = checkpointer.restore(
            self.checkpoint_dir_path,
            item=self.item_spec,
            restore_args=self.restore_args,
        )
        self.normalizer_stats = normalizer_stats
        # Handle both dict and PyTree for policy
        self.policy_params = (
            policy_tree["params"]
            if isinstance(policy_tree, dict) and "params" in policy_tree
            else policy_tree
        )
        self.value_params = value_tree
        self.p_keys = self._hidden_keys(self.policy_params)
        self.det_obs = int(self.policy_params[self.p_keys[0]]["kernel"].shape[0])
        self.det_act = int(self.policy_params[self.p_keys[-1]]["bias"].shape[0])
        print(f"[info] obs_dim={self.det_obs}, act_dim={self.det_act}")
        print(
            f"[info] policy hidden sizes="
            f"{[int(self.policy_params[k]['bias'].shape[0]) for k in self.p_keys[:-1]]} -> {self.det_act}"
        )
        print("final bias shape:", self.policy_params[self.p_keys[-1]]["bias"].shape)


    @staticmethod
    def _hidden_keys(params_dict):
        """Extract and sort hidden_* keys from a params dict.

        Args:
            params_dict: Dictionary of parameters.

        Returns:
            List of sorted hidden_* keys.
        """
        ks = [k for k in params_dict.keys() if k.startswith("hidden_")]
        ks.sort(
            key=lambda s: int(s.split("_")[1])
            if "_" in s and s.split("_")[1].isdigit() else 10**9
        )
        return ks

    def _to_row(self, x):
        """Convert 1D array to 2D row vector if needed.

        Args:
            x: Input array.

        Returns:
            2D array (row vector).
        """
        x = jnp.asarray(x)
        return x[None, :] if x.ndim == 1 else x

    def _normalize_obs(self, x: jnp.ndarray, stats: dict) -> jnp.ndarray:
        """Apply Z-score normalization to observation.

        Args:
            x: Observation array.
            stats: Dict with 'mean' and 'std'.

        Returns:
            Normalized observation.
        """
        mean = jnp.asarray(stats.get("mean", 0.0), dtype=jnp.float32)
        std = jnp.asarray(stats.get("std", 1.0), dtype=jnp.float32)
        x = self._to_row(x)
        mean = self._to_row(mean)
        std = self._to_row(std)
        eps = 1e-8
        return (x - mean) / (std + eps)

    def _apply_dense(self, h, w, b, nonlinearity=True):
        """Apply dense (fully connected) layer with optional tanh activation.

        Args:
            h: Input array.
            w: Weight matrix.
            b: Bias vector.
            nonlinearity: Whether to apply tanh activation.

        Returns:
            Output after dense layer (and activation if specified).
        """
        h = jnp.dot(h, w) + b
        return jnp.tanh(h) if nonlinearity else h

    def _policy_forward(self, x: jnp.ndarray) -> jnp.ndarray:
        """Forward pass for policy MLP (normalized input to action output).

        Args:
            x: Input observation array.

        Returns:
            Action output array.
        """
        h = self._normalize_obs(x, self.normalizer_stats)
        for i, k in enumerate(self.p_keys):
            w = self.policy_params[k]["kernel"]
            b = self.policy_params[k]["bias"]
            nonlin = i < len(self.p_keys) - 1
            h = self._apply_dense(h, w, b, nonlinearity=nonlin)
        mean = h[..., :self.act_dim]
        return mean

    def convert_to_onnx(self):
        """
        Public API: Restore checkpoint, convert to ONNX, and check output.
        """
        self._restore_checkpoint()

        tf_policy_tfcallable = jax2tf.convert(self._policy_forward,
                                              with_gradient=False, enable_xla=False)
        policy_signature = [tf.TensorSpec(shape=(1, self.det_obs),
                                          dtype=tf.float32, name="obs")]
        tf_policy_fn = tf.function(tf_policy_tfcallable,
                                   autograph=False, input_signature=policy_signature)
        cf_policy = tf_policy_fn.get_concrete_function()

        gdef = cf_policy.graph.as_graph_def(add_shapes=True)

        num_replaced = self._replace_ops_in_graphdef(
            gdef,
            {"PreventGradient": "Identity", "StopGradient": "Identity"}
        )

        print(f"[info] replaced ops in GraphDef: {num_replaced}")

        input_names  = [t.name for t in cf_policy.inputs]
        output_names = [t.name for t in cf_policy.outputs]
        print("[info] inputs:", input_names, "outputs:", output_names)

        _, _ = tf2onnx.convert.from_graph_def(
            gdef,
            input_names=input_names,
            output_names=output_names,
            opset=17,
            output_path=self.onnx_file_path,
        )
        print(f"[OK] Saved policy ONNX: {self.onnx_file_path}")

        self._check_policy_outputs_close()

    @staticmethod
    def _replace_ops_in_graphdef(gdef_, replace_map):
        """Replace specified ops in TensorFlow GraphDef for ONNX export compatibility.

        Args:
            gdef_: TensorFlow GraphDef object.
            replace_map: Dict mapping op names to replacements.

        Returns:
            Number of ops replaced.
        """
        replaced = 0
        for n in gdef_.node:
            if n.op in replace_map:
                n.op = replace_map[n.op]
                replaced += 1
        if gdef_.library and gdef_.library.function:
            for f in gdef_.library.function:
                for n in f.node_def:
                    if n.op in replace_map:
                        n.op = replace_map[n.op]
                        replaced += 1
        return replaced

    def _check_policy_outputs_close(self):
        """Validate ONNX output matches JAX output for random input (within tolerance).

        Raises:
            AssertionError: If ONNX and JAX outputs do not match within tolerance.
        """
        # Generate random input and compare outputs
        rng = np.random.RandomState(0)

        # predict with jax
        x = rng.randn(1, self.det_obs).astype(np.float32)
        y_jax = np.asarray(self._policy_forward(x))

        # predict with onnx
        sess = ort.InferenceSession(
            self.onnx_file_path, providers=["CPUExecutionProvider"])
        inp_name = sess.get_inputs()[0].name
        out_names = [o.name for o in sess.get_outputs()]
        print("ONNX input:", inp_name, "outputs:", out_names)

        y_onnx = sess.run(None, {inp_name: x})[0]
        print("Policy  JAX:", y_jax.shape, "ONNX:", y_onnx.shape)

        np.testing.assert_allclose(y_jax, y_onnx, rtol=1e-3, atol=1e-4)
        print("[OK] Policy ONNX output matches JAX within tolerance.")

ImportError: cannot import name 'runtime_version' from 'google.protobuf' (/home/sandbox/Work/mujoco_playground/learning/notebooks/venv/lib/python3.12/site-packages/google/protobuf/__init__.py)

In [19]:
checkpoint_dir_path = "/home/sandbox/Work/mujoco_playground/learning/notebooks/checkpoints/HunterJoystick-20251002-093320/checkpoints/000100270080"
onnx_file_path = "/home/sandbox/Work/mujoco_playground/learning/notebooks/onnx/output.onnx"
policy_hidden_layer_sizes = tuple([128, 128, 128, 128])
value_hidden_layer_sizes = tuple([256, 256, 256, 256, 256])
obs_dim = 41
act_dim = 10
converter = OnnxConvert(
    checkpoint_dir_path,
    onnx_file_path,
    obs_dim=obs_dim,
    act_dim=act_dim,
    policy_hidden_layer_sizes=policy_hidden_layer_sizes,
    value_hidden_layer_sizes=value_hidden_layer_sizes
)
converter.convert_to_onnx()

NameError: name 'OnnxConvert' is not defined

In [ ]:
import onnxruntime


class OnnxInfer:
    def __init__(self, onnx_model_path, input_name="obs", awd=False):
        self.onnx_model_path = onnx_model_path
        self.ort_session = onnxruntime.InferenceSession(
            self.onnx_model_path, providers=["CPUExecutionProvider"]
        )
        self.input_name = input_name
        self.awd = awd

    def infer(self, inputs):
        if self.awd:
            outputs = self.ort_session.run(None, {self.input_name: [inputs]})
            return outputs[0][0]
        else:
            outputs = self.ort_session.run(
                None, {self.input_name: inputs.astype("float32")}
            )
            return outputs[0]

In [15]:
import time
obs_size = 41
# onnx_model_path = "/home/onnx/output.onnx"
oi = OnnxInfer(onnx_file_path, awd=True, input_name="obs:0")
times = []
for i in range(1000):
    inputs = np.random.uniform(size=obs_size).astype(np.float32)
    # inputs = np.arange(obs_size).astype(np.float32)
    # print(inputs)
    start = time.time()
    print("Output: ")
    print(oi.infer(inputs))
    times.append(time.time() - start)

print("Average time: ", sum(times) / len(times))
print("Average fps: ", 1 / (sum(times) / len(times)))

NoSuchFile: [ONNXRuntimeError] : 3 : NO_SUCHFILE : Load model from /home/sandbox/Work/mujoco_playground/learning/notebooks/onnx/output.onnx failed:Load model /home/sandbox/Work/mujoco_playground/learning/notebooks/onnx/output.onnx failed. File doesn't exist